# jepa-as-a-tool: train the world model on a GPU

Runs the whole GPU part: data → V-JEPA 2 latents → translator → predictor.
At the end you download one small folder, `runs/wm`, and send it back.

**Kaggle settings (right panel):**
- Accelerator: **GPU T4 x1** (or P100)
- Internet: **on** (needs a verified phone number)
- Persistence: **Files only**, so `data/` survives a restart and you can skip finished steps

Works on Colab too: same cells, files go to `/content` instead of `/kaggle/working`.

In [ ]:
import os

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev 'https://github.com/RahulCHANDER25/jepa-as-a-tool.git' {REPO}
%cd {REPO}
!pip install -q -e .

If pip printed that it upgraded **numpy** or **torch**, restart the kernel (Run → Restart), then run the first cell again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git log --oneline -1

## 1. Data (CPU, ~2 min)

4,000 episodes of 6 random pushes in the real PushT physics = 28,000 poses.

In [ ]:
!jat gen data/random-4k --episodes 4000 --workers 4

## 2. V-JEPA 2 latents (GPU, ~10–20 min)

Renders every pose and encodes it with V-JEPA 2 ViT-L (1.3 GB download).
Writes `data/random-4k/latents.npy`, ~3.7 GB (64 tokens × 1024 per frame).

In [ ]:
!jat encode data/random-4k

## 3. Translator = the probe (GPU, ~5 min)

Learns latent → pose. **This answers: does V-JEPA 2 see where the T is?**
The last line prints the median error on test frames.

- Good: under ~5 px and ~3°. Go on.
- Bad: over ~15 px or ~10°. Stop and send me the output.

In [ ]:
!jat train translator data/random-4k --out runs/wm

## 4. Predictor = the world model (GPU, ~20–40 min)

Learns (latent, push) → next latent. It prints the time left as it trains, and saves the best
checkpoint. At the end it prints a table: imagined pose vs true pose after 0, 1, 2, ... pushes.
Compare with the `no-move` columns (the error if the model always said "nothing moves").

In [ ]:
!jat train predictor data/random-4k --out runs/wm

## 5. Send back

Download `wm.zip` (Output panel on Kaggle, Files panel on Colab) and send it,
plus a copy of the output of steps 3 and 4.

In [ ]:
!cd runs && zip -r {WORK}/wm.zip wm
!ls -lh {WORK}/wm.zip